In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [ ]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

device_reranker = utils.device

In [ ]:
def compute(dict_data, reranker, dir_reranker):
    # flag
    dict_flag = dict_data["flag"]
    
    # queries
    dict_text = dict_data["profile"]
    for text_type in ["concat", "separate"]:
        dict_text.update(dict_data[text_type])
    
    # documents
    d_documents = dict_data["items"]["candidates"]
    
    # compute reranking score
    for text_type, d_query in dict_text.items():
        t = text_type.replace("concat_", "").replace("separate_", "")
        d_flag = dict_flag[t]
        ner = dict_data["ner"]
        path_reranker = f"{dir_reranker}/{text_type}_{utils.rename(reranker.model_name)}_inst{reranker.inst_type}_{ner}.pickle"
        dict_score = reranker.load(path_reranker, d_query, d_flag, d_documents)

def run(reranker, data_name, N_icl=[1,3,5], flag_replace_NER=False):
    from src.data_loader import Loader
    loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)
    dict_data = loader.load_data()

    dir_reranker = f"{dir_workspace}/reranking_data/{version_exp}/{data_name}"
    os.makedirs(dir_reranker, exist_ok=True)    
    
    d_inst = utils.load_inst(reranker.model_name, flag_replace_NER=flag_replace_NER)
    for inst_type, inst_text in d_inst.items():
        print(f"{utils.rename(reranker.model_name):30} {data_name:30} {inst_type:10} NER{flag_replace_NER}")
        reranker.set_instruction_text(inst_type, inst_text)
        compute(dict_data, reranker, dir_reranker)

In [ ]:
data_names = ["MovieLens", "Job"] + [f"ARD_{a}" for a in [
    "CDs_and_Vinyl", "Movies_and_TV", "Toys_and_Games", "Sports_and_Outdoors"
]]

flag_replace_NER = [False, True][0]
N_icl = [1,3,5]

model_names_reranker = [
    "BM25",
    "BAAI/bge-reranker-v2-m3",
    "Alibaba-NLP/gte-reranker-modernbert-base",
    "Qwen/Qwen3-Reranker-0.6B",
    "Qwen/Qwen3-Reranker-8B"
]
for model_name_reranker in model_names_reranker:
    from src.reranker import Reranker
    model_id = f"{dir_parent}/models/reranker_models/{model_name_reranker}"
    reranker = Reranker(model_id, device_reranker)
    for data_name in data_names:
        run(reranker, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)